# onnxsim CUDA feature tests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnxsim/onnxsim/blob/master/examples/cuda_feature_tests/cuda_feature_tests.ipynb)

A hands-on notebook exercising onnxsim's CUDA-related features against a real GPU:

- the `providers` / `--cuda` execution-provider plumbing (`onnxsim.simplify`, `onnxsim.backend.run_model`, the CLI)
- the `(name, options)` tuple form, e.g. pinning `device_id`
- the DLPack zero-copy path for CUDA `torch.Tensor` inputs (`onnxsim.backend.as_ort_value` / `Runner.run_with_ort_values`)
- provider-validation errors (an unavailable provider fails loudly instead of silently falling back to CPU)
- `providers` threaded through `onnxsim.accuracy.measure_accuracy_drop`

This is meant to be run **by hand**, on demand -- against Colab or any other machine with an NVIDIA GPU -- not wired into an unattended nightly trigger. (Colab's own terms of service restrict automating it as a scheduled/background CI backend; see the discussion that led to this notebook.)

**Before running:** in Colab, pick a GPU runtime -- `Runtime → Change runtime type → T4 GPU` (or any NVIDIA GPU) -- then run the cells top to bottom. Outside Colab, just run it in any Python environment with an NVIDIA GPU and driver.

## 0. Confirm the runtime actually has a GPU

In [ ]:
!nvidia-smi

## 1. Install onnxsim + the GPU build of onnxruntime

By default this installs the latest development build of onnxsim from
[TestPyPI](https://test.pypi.org/project/onnxsim/) -- a pre-built wheel published
from `master` daily by CI (see `scripts/gc_testpypi.py` and the `upload_pypi` job in
`.github/workflows/build-and-test.yml`), so no C/C++ toolchain is needed in the Colab
runtime. The exact dev version is resolved and pinned at install time (see the cell
below for why an unpinned install can silently resolve to the older *real* PyPI
release instead). Set `BUILD_FROM_SOURCE = True` below to instead build a specific
`BRANCH`/tag from source -- e.g. to exercise unreleased changes before they reach
master. The source build does **not** build ONNX Runtime itself (see `CLAUDE.md`), so
it stays reasonably quick, but it does require a working C++20 toolchain and
CMake >= 3.22.

In [ ]:
import json
import pathlib
import urllib.request

BUILD_FROM_SOURCE = False  # True to build BRANCH from source instead of TestPyPI
BRANCH = "master"

if BUILD_FROM_SOURCE:
    if not pathlib.Path("onnxsim_repo").exists():
        !git clone --branch {BRANCH} --depth 1 https://github.com/onnxsim/onnxsim.git onnxsim_repo
    %cd onnxsim_repo
    # No [onnxruntime] extra: that pulls in the CPU onnxruntime package, which
    # installs into the same "onnxruntime" import namespace as onnxruntime-gpu
    # below -- having both installed at once is a known source of broken/
    # inconsistent onnxruntime installs. onnxruntime-gpu alone satisfies
    # onnxsim's `import onnxruntime`.
    !pip install -e .
    %cd ..
else:
    # Resolve and PIN the exact latest TestPyPI dev build ourselves, rather than
    # letting pip pick a version across both indexes: with --extra-index-url
    # pulling in real PyPI (needed for onnxsim's other dependencies, since
    # TestPyPI doesn't mirror the full package index), pip's resolver considers
    # onnxsim candidates from BOTH indexes together and picks the highest PEP 440
    # version overall -- and a plain release like "0.7.3" on real PyPI always
    # outranks any "0.7.3.devN" prerelease on TestPyPI of the same base version,
    # even with --pre. Left unpinned, that silently installs the *real*, older
    # stable release instead of the intended TestPyPI dev build.
    releases = json.load(
        urllib.request.urlopen("https://test.pypi.org/pypi/onnxsim/json")
    )["releases"]
    dev_versions = [v for v in releases if ".dev" in v and releases[v]]
    latest_dev = max(dev_versions, key=lambda v: int(v.rsplit(".dev", 1)[1]))
    print(f"Installing onnxsim=={latest_dev} from TestPyPI")
    # No [onnxruntime] extra here either -- same reason as the source-build branch
    # above.
    !pip install --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "onnxsim=={latest_dev}"

# onnxruntime-gpu >=1.27 requires CUDA 13 (nvidia-cuda-runtime~=13.0 etc.), but
# as of this writing NVIDIA has not published a real CUDA-13 cuBLAS wheel:
# the unsuffixed "nvidia-cublas" package's version number reads 13.x, but it
# still only ships libcublasLt.so.12, and "nvidia-cublas-cu13" is an unrelated
# 0.0.1 placeholder -- so libcublasLt.so.13 can't currently be satisfied via
# pip at all. Pin to onnxruntime-gpu==1.26.0, the last release built against
# CUDA 12 (with the full complement of mature nvidia-*-cu12 wheels actually
# available); the GPU driver is backward compatible with CUDA 12 binaries.
# Bump this once a real CUDA-13 cuBLAS wheel ships.
# onnxruntime-gpu's own [cuda]/[cudnn] extras also miss cuBLAS itself even
# though the CUDA provider's .so links against it, so install it explicitly.
!pip install -q "onnxruntime-gpu[cuda,cudnn]==1.26.0" nvidia-cublas-cu12

# The nvidia-* pip packages above drop their .so files under
# .../site-packages/nvidia/<name>/lib/, which is not a standard linker search
# path. onnxruntime's CUDA provider does a bare dlopen("libcublasLt.so.12")
# etc. with no RPATH pointing there (unlike PyTorch's own wheels, which do set
# one up) -- so the libraries being installed isn't enough; LD_LIBRARY_PATH has
# to actually include those directories before the provider is first loaded.
import glob
import os
import sys

nvidia_lib_dirs = sorted({
    d
    for base in sys.path
    for d in glob.glob(os.path.join(base, "nvidia", "*", "lib"))
})
os.environ["LD_LIBRARY_PATH"] = ":".join(
    nvidia_lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")]
)
print("Added to LD_LIBRARY_PATH:", nvidia_lib_dirs)

## 2. Confirm onnxruntime sees the CUDA provider

In [ ]:
import onnxruntime as rt

import onnxsim
from onnxsim import backend

available = rt.get_available_providers()
print("Available onnxruntime providers:", available)
assert backend.has_onnxruntime()
assert "CUDAExecutionProvider" in available, (
    "CUDAExecutionProvider is not available -- pick a GPU runtime "
    "(Runtime > Change runtime type) and re-run from the top."
)
print("CUDA provider available.")

## 3. Test helpers

One small model, built with `onnx.parser` (per this repo's own test convention -- see `CLAUDE.md`), whose `a + b` onnxsim can constant-fold, plus a `record()` helper that prints and tracks pass/fail so the summary at the end can assert all of them passed.

In [ ]:
import numpy as np
import onnx
from onnx import parser

RESULTS = {}


def record(name, ok, detail=""):
    RESULTS[name] = ok
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}" + (f" -- {detail}" if detail else ""))
    if not ok:
        raise AssertionError(f"{name}: {detail}")


def foldable_model(a_value=1.0):
    model = parser.parse_model(
        f"""
        <ir_version: 10, opset_import: ["": 18]>
        foldable (float[2,2] x) => (float[2,2] y)
        <float[2,2] a = {{{a_value}, {a_value}, {a_value}, {a_value}}}, float[2,2] b = {{2.0, 2.0, 2.0, 2.0}}>
        {{
          c = Add(a, b)
          y = Add(c, x)
        }}
        """
    )
    onnx.checker.check_model(model)
    return model


X = np.arange(4, dtype=np.float32).reshape(2, 2)
CUDA = ["CUDAExecutionProvider", "CPUExecutionProvider"]

## Test A -- `backend.run_model`: CPU vs. CUDA parity

In [ ]:
cpu_out = backend.run_model(
    foldable_model(), {"x": X}, providers=["CPUExecutionProvider"]
)
cuda_out = backend.run_model(foldable_model(), {"x": X}, providers=CUDA)
ok = np.allclose(cpu_out["y"], cuda_out["y"]) and np.allclose(cuda_out["y"], X + 3.0)
record(
    "backend.run_model CPU/CUDA parity",
    ok,
    f"cpu={cpu_out['y'].tolist()} cuda={cuda_out['y'].tolist()}",
)

## Test B -- `onnxsim.simplify(providers=CUDA)`: constant folding on the GPU

In [ ]:
opt_cpu, ok_cpu = onnxsim.simplify(foldable_model(), check_n=3)
opt_cuda, ok_cuda = onnxsim.simplify(foldable_model(), check_n=3, providers=CUDA)

folded_on_gpu = len(opt_cuda.graph.node) == 1
same_result = np.allclose(
    backend.run_model(opt_cpu, {"x": X})["y"],
    backend.run_model(opt_cuda, {"x": X})["y"],
)
record(
    "simplify() folds on CUDA and matches CPU",
    ok_cpu and ok_cuda and folded_on_gpu and same_result,
    f"nodes_after_cuda_fold={len(opt_cuda.graph.node)}",
)

## Test C -- `(name, options)` tuple form: pinning `device_id`

In [ ]:
providers_pinned = [("CUDAExecutionProvider", {"device_id": 0}), "CPUExecutionProvider"]
out = backend.run_model(foldable_model(), {"x": X}, providers=providers_pinned)
record("device_id tuple form", np.allclose(out["y"], X + 3.0))

## Test D -- CLI `--cuda` end to end

In [ ]:
import subprocess

onnx.save(foldable_model(), "cuda_test_in.onnx")
result = subprocess.run(
    ["onnxsim", "cuda_test_in.onnx", "cuda_test_out.onnx", "--cuda"],
    capture_output=True,
    text=True,
)
print(result.stdout[-1500:])
cli_model = onnx.load("cuda_test_out.onnx")
cli_out = backend.run_model(cli_model, {"x": X})
record(
    "CLI --cuda",
    result.returncode == 0
    and len(cli_model.graph.node) == 1
    and np.allclose(cli_out["y"], X + 3.0),
    result.stderr[-500:] if result.returncode != 0 else "",
)

## Test E -- an unavailable provider fails loudly instead of silently degrading to CPU

In [ ]:
try:
    backend.run_model(foldable_model(), {"x": X}, providers=["ROCMExecutionProvider"])
    record("unavailable provider raises", False, "expected ValueError, none raised")
except ValueError as e:
    record("unavailable provider raises", "not available" in str(e), str(e))

## Test F -- DLPack zero-copy path with a CUDA `torch.Tensor`

Colab's GPU runtime ships `torch` with CUDA support preinstalled, so no extra install is needed here.

In [ ]:
import torch

from onnxsim.backend import Runner, as_ort_value

assert torch.cuda.is_available(), "torch does not see a CUDA device"

runner = Runner(foldable_model(), providers=CUDA)
x_gpu = torch.from_numpy(X).cuda()
ort_outputs = runner.run_with_ort_values({"x": as_ort_value(x_gpu)})
y = ort_outputs["y"].numpy()
record("DLPack zero-copy CUDA tensor", np.allclose(y, X + 3.0))

## Test G -- `providers` threaded through `measure_accuracy_drop`

In [ ]:
from onnxsim.accuracy import measure_accuracy_drop

report = measure_accuracy_drop(
    foldable_model(a_value=1.0),
    foldable_model(a_value=1.01),  # stand-in for a quantized model: same graph, perturbed weight
    calibration_data=[{"x": X}],
    providers=CUDA,
)
print(report)
ok = report.all_finite and report.per_output["y"].max_abs_error > 0
record("measure_accuracy_drop threads CUDA providers", ok, str(report))

## Summary

In [ ]:
print("=" * 48)
for name, ok in RESULTS.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {name}")
assert all(RESULTS.values()), "one or more CUDA feature tests failed"
print(f"\nAll {len(RESULTS)} CUDA feature tests passed.")